---
title: "À quelques kilomètres près : lycées et géographie des cadres dans trois aires urbaines de province"
output-file: index.html
---

In [ ]:
# Le SSPCloud définit MPLBACKEND=Agg : sans cette ligne, les figures ne s'affichent pas
%matplotlib inline
import warnings

import pandas as pd
from IPython.display import display

from cadres_lycees import (
    analyse,
    cartes,
    collecte,
    config,
    figures,
    geo,
    h1,
    h3,
    nettoyage,
    pipeline,
    robustesse,
    tableaux,
)

# Sans BUCKET_S3, stockage.py avertit qu'il lit et écrit dans data/ : c'est le cas prévu d'un
# compte neuf (README). Les autres avertissements, dont le repli d'une copie gelée, restent
warnings.filterwarnings("ignore", message="Variables d'environnement manquantes")
# pandas coupe les cases à 50 caractères : les intervalles et les sha256 seraient tronqués
pd.set_option("display.max_colwidth", None)


def afficher_onglets(morceaux):
    """Imprime le balisage Quarto des onglets et affiche les cartes, dans l'ordre."""
    for morceau in morceaux:
        if isinstance(morceau, str):
            print(morceau)
        else:
            display(morceau)

## Résumé

Dans les aires urbaines de Nantes, Lyon et Strasbourg, les communes proches d'un lycée privé ou d'un lycée à forte valeur ajoutée comptent-elles plus de cadres que leurs voisines ? Pour répondre, nous avons construit une base de 789 communes qui croise le recensement, l'annuaire de l'éducation, la valeur ajoutée des lycées et quatre autres infrastructures.

Entre communes contiguës, être plus près d'un lycée privé ne s'accompagne pas d'un écart net de part de cadres. À Lyon, le lien est proche de zéro : pour 5 km de moins jusqu'au lycée privé, l'écart de part de cadres est compris entre −1,0 et +2,7 points. À Nantes et à Strasbourg, les données ne permettent pas de conclure. Entre 2011 et 2022, la part de cadres a davantage augmenté là où les lycées proches sont nombreux que là où leur valeur ajoutée est élevée, mais le test utilisé désavantage la qualité (4.4).

Il s'agit d'associations : un lycée privé peut aussi s'implanter là où vivent déjà des cadres.

## 1. Introduction

### 1.1 Pourquoi les lycées

Les familles paient pour l'école de leurs enfants, et ce prix se lit sur le marché du logement. À Paris, Fack et Grenet (2010) comparent les prix immobiliers dans les secteurs des collèges publics : un collège de secteur plus performant d'un écart-type va de pair avec des logements plus chers de 1,4 à 2,4 %. Cet écart se réduit quand un établissement privé est proche, car le privé offre une solution de repli à ceux qui veulent échapper à la carte scolaire. Black (1999) mesurait déjà ce lien aux États-Unis en comparant des logements situés de part et d'autre d'une limite de secteur scolaire ; nos paires de communes contiguës reprennent cette idée sous une forme simplifiée.

La question est d'actualité. Depuis la fin du mois de septembre 2026, des lycéens manifestent et bloquent leurs établissements dans toute la France contre les classes surchargées, les enseignants non remplacés et l'état des locaux. Début octobre, plus d'un millier de lycées sur environ 3 700 étaient touchés, selon le ministère de l'Intérieur. Ce mouvement rappelle que les conditions d'études varient fortement d'un lycée à l'autre. Les familles qui le peuvent répondent à ces écarts de deux façons : en choisissant le privé, ou en s'installant près des lycées qu'elles jugent meilleurs. Si le second comportement est fréquent, il devrait laisser une trace dans la géographie des cadres.

Nous transposons donc la question aux lycées de province, dont la valeur ajoutée est publiée chaque année. Le poids du privé y varie fortement. En métropole, environ 78 % des élèves du second degré sont scolarisés dans le public, contre environ 60 % dans les académies de Nantes et de Rennes (DEPP, *Repères et références statistiques*). Dans l'Ouest, le privé est donc un choix courant, et sans doute moins sélectif socialement. En comparant une aire de l'Ouest à deux autres aires, on peut voir si le lien entre lycées et cadres en dépend.


### 1.2 Question et hypothèses

**Question.** À position égale dans l'aire, les communes qui offrent un meilleur accès à des lycées privés ou à forte valeur ajoutée comptent-elles plus de cadres que leurs voisines, et ce lien tient-il face aux autres infrastructures ?

| | Hypothèse | Comparaison principale | Statut |
|---|---|---|---|
| H1 | À distance égale du centre, plus de cadres là où l'accès aux lycées privés sous contrat est meilleur | Paires de communes contiguës (4.1), anneaux de distance (4.2) | verdict par aire |
| H2 | Ce lien est plus faible dans l'aire de l'Ouest (Nantes), où le privé est plus répandu | H1 aire par aire (4.3) | descriptive |
| H3 | La qualité des lycées accessibles, plus que leur nombre, accompagne la hausse de la part de cadres de 2011 à 2022 | Régression sur la qualité et le nombre (4.4) | verdict, 3 aires ensemble |
| H4 | Le lien est plus fort avec les lycées qu'avec les gares, les médecins et les équipements | Mêmes paires pour chaque infrastructure (5) | descriptive |


### 1.3 Ce que le projet peut et ne peut pas montrer

Le projet décrit des associations. Quatre limites empêchent d'aller plus loin :

- **causalité inverse** : un lycée privé peut s'implanter ou se maintenir là où vivent déjà des familles aisées ;
- **sélection résidentielle** : les préférences des ménages ne sont pas observées ;
- **croissance démographique** : elle peut précéder l'ouverture de lycées et accompagne l'arrivée de cadres ; elle n'est pas contrôlée ;
- **mesure de l'accès** : distances à vol d'oiseau depuis le centroïde de la commune.

Lorsque nous ne trouvons pas de lien, nous indiquons quelles tailles d'association l'intervalle de confiance permet d'exclure.

## 2. Construire la base : une ligne par commune, quatre familles de variables

### 2.1 Schéma du pipeline et tableau des sources

Le rapport reconstruit la base à chaque exécution à partir des fichiers bruts. Certaines sources sont mises à jour en continu : nous les avons téléchargées une fois et conservées dans une copie datée (« copie gelée »), pour que les résultats restent identiques d'une exécution à l'autre. L'empreinte `sha256` de chaque fichier est vérifiée à la lecture (annexe), et chaque étape du traitement est une fonction de `src/cadres_lycees/`.

```{mermaid}
flowchart LR
    A[Communes<br/>aires, contours] --> D[Accessibilité<br/>une fonction pour tous les points]
    C[Lycées<br/>annuaire, IPS, IVAL] --> D
    E[Gares, médecins,<br/>sport, culture] --> D
    B[Recensement<br/>cadres 2011 et 2022] --> G[Base communale]
    D --> G
    F[Contrôles<br/>distance au centre, densité, prix] --> G
    G --> P[Paires de communes contiguës]
```

| Source | Producteur | Accès | Usage |
|---|---|---|---|
| Recensement 2022 (évolution et structure de la population, séries historiques) | Insee | fichiers | cadres 2011 et 2022, densité |
| Annuaire de l'éducation | Éducation nationale | API, copie gelée | lycées, secteur, coordonnées |
| IPS et IVAL des lycées | DEPP | API, copies gelées | composition sociale, valeur ajoutée |
| Aires d'attraction des villes, table de passage | Insee | fichiers | périmètre, codes communes |
| Contours des communes et des IRIS | IGN | paquet Python `cartiflette`, copies gelées | géométries, contiguïté |
| Gares, APL, Data ES, BPE 2025 | SNCF, DREES, Sports, Insee | API et fichiers, copies gelées | H4 |
| Statistiques DVF | DGFiP, Etalab | fichier, copie gelée | prix au m² (robustesse) |

: Sources de données du projet {#tbl-sources}

In [ ]:
# La base est reconstruite depuis les copies gelées ; final/ n'est qu'une sortie
base, lycees, paires = pipeline.construire_base(config.AIRES_RETENUES)

In [ ]:
#| label: tbl-effectifs
#| tbl-cap: "Effectifs aux étapes clés de la construction de la base, par aire."
tableaux.tableau_effectifs(analyse.effectifs_base(base, lycees, paires))

Nous avons vérifié à chaque étape qu'aucune commune ni aucun lycée n'était perdu en route. Les 153 lycées retenus dans les trois aires font partie des 2 413 lycées de métropole utilisés pour mesurer l'accès, car une commune en bordure d'aire peut avoir son lycée le plus proche à l'extérieur. Le prix au m² manque presque partout dans l'aire de Strasbourg, parce que les ventes d'Alsace-Moselle ne figurent pas dans la base des ventes immobilières (DVF).

### 2.2 Variable à expliquer

Nous mesurons la part des cadres et professions intellectuelles supérieures parmi les actifs de 25 à 54 ans (actifs occupés et chômeurs ayant déjà travaillé), en 2011 et en 2022, ainsi que son évolution en points. Cette tranche d'âge correspond à celle des parents de lycéens et écarte les retraités. Le recensement donne les arrondissements de Lyon, avec la même définition aux deux dates. H1, H2 et H4 portent sur 2022, H3 sur l'évolution.


### 2.3 Variable d'intérêt : l'accessibilité aux lycées

Une commune sans lycée peut en avoir un à quelques kilomètres. Nous mesurons donc l'accès aux lycées plutôt que leur présence dans la commune. Nous retenons les lycées généraux, technologiques et polyvalents, publics et privés sous contrat, et calculons trois mesures à vol d'oiseau depuis le centroïde de chaque commune :

- la distance au lycée privé le plus proche (mesure principale de H1) ;
- le nombre de lycées privés à moins de 5 km (mesure secondaire) ;
- la valeur ajoutée moyenne des lycées à moins de 5 km, pondérée par leurs candidats au bac. Cet indicateur du ministère (IVAL) compare le taux d'accès de la seconde au bac observé à celui qu'on attendrait au vu du profil des élèves ; nous utilisons sa moyenne sur les sessions 2012 à 2022.

Le calcul utilise les 2 413 lycées retenus en métropole, y compris ceux situés hors des trois aires, pour que les distances ne soient pas faussées au bord des aires.

In [ ]:
#| label: tbl-acces
#| tbl-cap: "Accès des communes aux lycées et aux autres infrastructures, par aire."
tableaux.tableau_acces(analyse.resume_acces(base))

Le privé est bien moins accessible à Strasbourg. Seules 4 % des communes y ont un lycée privé à moins de 5 km, contre 16 % à Nantes et 23 % à Lyon, et la commune médiane en est à 16 km. Le lycée public est, en médiane, à 6 à 8 km dans les trois aires. À Strasbourg, le petit nombre de communes proches d'un lycée privé limitera la précision des estimations.

### 2.4 Contrôles et mauvais contrôles

Les comparaisons se font à l'intérieur d'une aire, avec deux contrôles : la distance au centre de l'aire et le logarithme de la densité. Plus une commune est loin du centre, plus elle est loin d'un lycée privé ; plus elle est dense, plus elle en est proche. Or la part de cadres est bien plus élevée au centre des aires qu'en périphérie (3.2). Sans ces contrôles, on comparerait surtout des communes centrales à des communes périphériques, au lieu de comparer des communes plus ou moins proches d'un lycée privé.

Le niveau de vie, l'IPS des lycées (indice de position sociale, calculé à partir des professions des parents d'élèves) et la croissance de la population ne servent jamais de contrôles, car ils dépendent en partie de la part de cadres elle-même. Le prix au m², qui influence la part de cadres autant qu'il en dépend, n'est utilisé que dans un test de robustesse.


### 2.5 Encadré « choix et biais »

::: {.callout-note title="Choix et biais"}
- **Nomenclature.** Le recensement de 2022 code les professions en PCS 2020, celui de 2011 en PCS 2003. Selon l'Insee, les deux premiers niveaux de la nomenclature, dont le groupe des cadres, n'ont pas changé, mais nous n'avons pas vérifié les reclassements de professions détaillées. Seule H3, qui compare 2011 et 2022, est concernée.
- **Valeurs manquantes.** Toutes les communes des trois aires ont des actifs et une part de cadres connue. Les autres manques sont traités un par un. La valeur ajoutée manque pour 12 communes de la population de H3, regroupées dans une catégorie à part. Le prix au m² n'est pas publié sous 10 ventes, ni pour l'Alsace-Moselle. Enfin, 7 lycées hors contrat n'ont pas de coordonnées et sont écartés de la robustesse qui ajoute le privé hors contrat. Aucune autre donnée ne manque.
- **Valeur ajoutée.** C'est une mesure relative, qui situe un lycée par rapport à ce qu'on attend de ses élèves, et elle varie d'une session à l'autre. Nous utilisons donc sa moyenne de 2012 à 2022, période qui sépare les deux recensements. Sa fiabilité au niveau des communes vaut 0,608 (@tbl-fiabilite), au-dessus du seuil de 0,5 que nous avions retenu : la mesure est assez fiable pour tester H3.
- **Prix.** Les statistiques de ventes immobilières (DVF) couvrent la période de juillet 2021 à juin 2026, après le recensement de 2022, et n'incluent pas l'Alsace-Moselle. La robustesse au prix ne porte donc que sur Lyon et Nantes.
- **Lycées étrangers.** L'annuaire ne contient pas les lycées situés à l'étranger. Or 55 communes de l'aire de Strasbourg sont à moins de 10 km de la frontière ; nous testons leur exclusion.
- **Lycées agricoles.** Ils sont exclus partout, car leur contrat avec le ministère de l'Agriculture n'entre pas dans nos critères. La Loire-Atlantique en compte 15 dans le privé, ce qui limite la portée de H2.
- **Distances à vol d'oiseau depuis le centroïde.** Nous testons aussi des distances calculées depuis la mairie.
- **Lyon.** La ville est découpée en arrondissements partout, sauf dans la carte à curseur (3.2).
- **Petites communes.** Sous 200 actifs, la part de cadres, estimée sur un échantillon, est imprécise. Aucune commune n'est dans ce cas à Nantes, contre 33 à Lyon et 72 à Strasbourg (@tbl-effectifs) ; nous testons leur exclusion.
:::

## 3. Trois aires, trois profils

### 3.1 Pourquoi ces trois aires

Nous avons comparé huit aires candidates sur six critères, sans regarder aucune donnée sur les cadres (tableau de support ci-dessous). Paris est exclue, car les transports en commun y jouent un rôle trop important.

- **(a) Une aire de l'Ouest.** Nous avons choisi Nantes plutôt que Rennes. Les deux n'ont qu'une cellule d'anneaux exploitable (voir 4.2), mais Nantes compte plus de paires de voisines dont une seule a un lycée privé à moins de 10 km (76 contre 71).
- **(b) Un contraste dans le poids du privé.** Le privé représente 47 % des lycées à Nantes, 45 % à Lyon et 27 % à Strasbourg, et l'écart d'IPS entre privé et public y est de 11,8, 23,7 et 22,8 points.
- **(c) Des tailles comparables.** Nantes compte 1,04 million d'habitants et Strasbourg 0,87 million. Lyon (2,33 millions) est la plus grande des huit aires, mais nous l'avons gardée pour le critère (e), jugé prioritaire : c'est la seule aire où les anneaux sont comparables à 5 km, donc la seule où l'on peut établir un résultat nul (4.2).
- **(d) Une dizaine de lycées privés.** Il y en a 18 à Nantes, 37 à Lyon et 9 à Strasbourg, juste sous le seuil.
- **(e) Assez de communes comparables.** La comparaison par anneaux (4.2) n'est possible à 5 km qu'à Lyon. À 10 km, elle l'est à Lyon, Nantes, Rennes et Strasbourg.
- **(f) La faisabilité.** Le recensement donne les arrondissements de Lyon.

Nous avons préféré Strasbourg à Bordeaux pour son contraste avec Nantes et parce que ses anneaux sont comparables à 10 km. Nous savions que cette aire serait fragile, avec seulement 11 communes exposées à 5 km.

In [ ]:
#| column: page
#| label: fig-aires
#| fig-cap: >-
#|   Part du privé sous contrat parmi les lycées et les candidats au bac, et IPS de chaque
#|   lycée par secteur, dans les 8 aires candidates (aires retenues en gras).
# La `fig_aires` montre les 8 candidates ; la base ne porte que sur les 3 aires retenues
referentiel_candidates = geo.referentiel_communes(config.AIRES_CANDIDATES)
lycees_8_aires = pipeline.lycees_avec_aire(referentiel_candidates)
profil = analyse.profil_aires(lycees_8_aires)
figures.fig_aires(
    profil,
    lycees_8_aires,
    "À Nantes, le privé est répandu et peu sélectif ; à Strasbourg, rare et sélectif",
    config.AIRES_RETENUES,
)

Le privé est aussi répandu à Nantes qu'à Lyon (47 % et 45 % des lycées), mais l'écart de position sociale entre privé et public y est deux fois plus faible (+12 contre +24 points). À Strasbourg, le privé est rare (16 % des candidats au bac) mais aussi sélectif qu'à Lyon. La prémisse de H2 se vérifie donc : dans l'Ouest, le privé est plus répandu et moins sélectif.

In [ ]:
#| output: asis
# Replié : le tableau de support justifie le choix sans alourdir la lecture
criteres_aires, support_aires, _ = analyse.lire_tableau_support()
print(
    tableaux.encadre_replie(
        "Tableau de support du choix des aires",
        tableaux.tableau_support(criteres_aires, support_aires),
    )
)

### 3.2 Le gradient centre-périphérie des cadres

La part de cadres baisse à mesure qu'on s'éloigne du centre. Comparer des communes proches et éloignées d'un lycée privé sans tenir compte de leur position reviendrait donc à comparer le centre à la périphérie.

In [ ]:
#| label: tbl-gradient
#| tbl-cap: "Part de cadres des 25-54 ans en 2022 par anneau de distance au centre."
# Le gradient centre-périphérie justifie de comparer les communes à distance égale du centre
tableaux.tableau_gradient(analyse.parts_par_anneau(base))

La part de cadres est 1,9 à 2,5 fois plus élevée à moins de 5 km du centre qu'au-delà de 30 km : 39 % contre 17 % à Lyon, 36 % contre 15 % à Nantes et 29 % contre 16 % à Strasbourg. La baisse n'est pas tout à fait régulière, puisqu'à Lyon l'anneau de 5 à 10 km (23 %) passe sous celui de 10 à 20 km (26 %). À Nantes et à Strasbourg, l'anneau de 0 à 5 km ne compte que 2 communes, la commune-centre et une voisine.

In [ ]:
#| output: asis
# Chaque recensement de la série historique est ramené au COG 2025
series_csp = {annee: collecte.serie_csp_2554(annee) for annee in config.GEOGRAPHIE_SERIE_CSP}
parts_dates = nettoyage.parts_cadres_dates(series_csp, collecte.passage_communes(), base)
cartes_dates = {
    aire: cartes.carte_cadres_dates(base[base["aire"] == aire], parts_dates, f"Aire de {aire}")
    for aire in config.AIRES_RETENUES
}
legende = (
    "**La part de cadres baisse du centre vers la périphérie à chaque recensement "
    "depuis 1999.** Part des cadres parmi les actifs de 25 à 54 ans par commune, de"
    " 1999 à 2022, une aire par onglet et un curseur pour l'année."
)
afficher_onglets(cartes.onglets_par_aire("fig-cadres-dates", cartes_dates, legende))

On retrouve ce gradient à chaque recensement depuis 1999, alors que la part de cadres augmente dans les trois aires. Cette carte décrit seulement le gradient : elle ne montre pas les lycées et ne permet pas de tester H1.

### 3.3 La géographie de l'accès aux lycées

In [ ]:
#| output: asis
cartes_acces = {
    aire: cartes.carte_acces(base[base["aire"] == aire], lycees, f"Aire de {aire}")
    for aire in config.AIRES_RETENUES
}
legende = (
    "**Hors des villes-centres, le lycée privé le plus proche est souvent à plus de"
    " 10 km.** Distance à vol d'oiseau du centroïde de chaque commune au lycée "
    "privé sous contrat le plus proche, lycées publics et privés en points, une "
    "aire par onglet."
)
afficher_onglets(cartes.onglets_par_aire("fig-acces", cartes_acces, legende))

Les lycées privés sont regroupés dans les villes-centres et dans quelques pôles secondaires. Au-delà, la distance augmente vite, surtout à Strasbourg, où la plupart des communes sont à plus de 10 km d'un lycée privé. Les communes exposées sont donc plus centrales que les autres, et la section 4 les compare à position égale.

## 4. À position égale, un meilleur accès aux lycées va-t-il de pair avec plus de cadres ?

### 4.1 Comparaison entre communes contiguës

Le résultat principal repose sur des **paires de communes contiguës**, c'est-à-dire deux communes de la même aire qui ont une frontière ou un point commun. On en compte 301 à Nantes, 1 127 à Lyon et 717 à Strasbourg. Deux voisines partagent le même marché de l'emploi et la même position dans l'aire, et ces facteurs disparaissent quand on fait la différence entre elles. Nous régressons l'écart de part de cadres entre les deux communes sur leur écart de distance au lycée privé le plus proche, en contrôlant les écarts de distance au centre et de densité. Si H1 est vraie, la pente est négative.

Comme une commune appartient à plusieurs paires, les paires ne sont pas indépendantes. Les intervalles de confiance viennent donc d'un **bootstrap par commune**, qui tire au hasard des communes et non des paires (1 000 tirages).

In [ ]:
#| column: page
#| label: fig-paires
#| fig-cap: >-
#|   Écart de part de cadres en 2022 et écart de distance au lycée privé le plus proche entre
#|   communes contiguës, à position égale, moyennes par classes et pente principale.
pentes_principales = h1.pentes_paires_par_aire(paires, "d_dist_lyc_prive_min")
figures.fig_paires(
    paires,
    pentes_principales,
    "Entre communes voisines, la pente est faible et son intervalle contient zéro partout",
)

La pente est négative dans les trois aires, mais faible, et les trois intervalles contiennent zéro. À position égale, une commune plus proche d'un lycée privé que sa voisine ne compte donc pas nettement plus de cadres.

::: {.callout-note title="Les pentes en détail"}
- Pour un km de distance au lycée privé en moins : −0,15 point de cadres à Nantes, −0,18 à Lyon et −0,06 à Strasbourg.
- À Lyon, l'intervalle [−0,53 ; +0,21] signifie qu'être 5 km plus près d'un lycée privé s'accompagne d'un écart compris entre −1,0 et +2,7 points de cadres.
- La borne haute de cet écart n'est pas négligeable. Du centre aux communes situées au-delà de 30 km, la part de cadres perd 22 points à Lyon (@tbl-gradient), soit de l'ordre d'un demi-point par km, et la borne basse de la pente (−0,53 point par km de distance au lycée privé) est du même ordre.
:::

Le « résultat nul établi » de Lyon (4.2) signifie donc que les données ne permettent pas de distinguer l'association de zéro. Il ne garantit pas qu'elle soit petite.

Le nuage de points est symétrique autour de zéro par construction : comme l'ordre des deux communes d'une paire est arbitraire, chaque paire y figure dans les deux sens.

### 4.2 Comparaison par anneaux et verdict de H1

La figure ci-dessous rassemble toutes les estimations utilisées pour le verdict de H1 : les pentes principale et secondaire des paires (4.1), puis les comparaisons par anneaux.

Dans chaque anneau de distance au centre (0-5, 5-10, 10-20, 20-30 et plus de 30 km), nous comparons les communes qui ont un lycée privé dans le rayon, dites exposées, aux autres, avec les contrôles de la section 2.4. Une cellule (aire × anneau) n'est estimée que si chaque groupe compte au moins 15 communes et si leurs distances moyennes au centre diffèrent de 2 km au plus. Sinon, nous fusionnons des anneaux voisins. Les intervalles viennent d'un bootstrap de 1 000 tirages des communes avec remise. Seules 6 cellules remplissent ces conditions : 2 à Lyon pour un rayon de 5 km et, pour un rayon de 10 km, 2 à Lyon, 1 à Nantes et 1 à Strasbourg (panneau de droite).

In [ ]:
#| column: page
#| label: fig-h1
#| fig-cap: >-
#|   Estimations de H1 par aire, avec intervalles à 95 % : pentes principale et secondaire des
#|   paires contiguës, écarts des cellules d'anneaux estimables à 5 et 10 km.
# Les deux pentes voient les mêmes tirages du bootstrap ; la `fig_h1` sert aussi H2
pentes_secondaires = h1.pentes_paires_par_aire(paires, "d_n_lyc_prive_5km")
anneaux = h1.ecarts_anneaux(base)
figures.fig_h1(
    pentes_principales,
    pentes_secondaires,
    anneaux,
    "Aucune pente des paires n'exclut zéro ; une seule cellule d'anneau l'exclut",
)

Les anneaux vont dans le même sens que les paires, sans trancher davantage. Les quatre cellules de Lyon sont positives, mais aucune n'exclut zéro, et une seule cellule, à Nantes, le fait.

::: {.callout-note title="Verdict de H1 par aire"}
- **Lyon : « infirmée » (résultat nul établi).** L'intervalle de la pente contient zéro, les deux anneaux comparés aux rayons de 5 et 10 km concordent et aucune cellule n'exclut zéro. Toutes les estimations de Lyon vont pourtant dans le sens de H1. Le mot « infirmée » désigne donc ici un lien trop faible pour être distingué de zéro, et non un lien de sens contraire.
- **Nantes et Strasbourg : « non concluante ».** L'intervalle contient aussi zéro, mais aucun anneau n'est estimable aux deux rayons pour confirmer ce résultat. On ne peut donc pas affirmer que le lien est nul.
:::

::: {.callout-note title="Le détail des anneaux et de la pente secondaire" collapse="true"}
- Les quatre cellules de Lyon donnent +1,5 à +3,1 points de cadres dans les communes exposées. Elles concordent entre les deux rayons, mais aucun de leurs intervalles n'exclut zéro.
- La cellule de Nantes au-delà de 20 km exclut zéro : +2,0 points [+0,1 ; +3,9].
- La pente secondaire, calculée sur le nombre de lycées privés à 5 km, a un intervalle qui contient zéro dans les trois aires (@fig-h1). Elle va dans le sens de H1 à Lyon (+0,41 point de cadres par lycée), mais dans le sens contraire à Nantes (−0,46) et à Strasbourg (−0,65). Dans ces deux aires, elle aurait de toute façon empêché de confirmer H1.
:::

### 4.3 L'Ouest est-il différent ?

Les pentes principales des trois aires sont très proches (@fig-h1) et leurs intervalles se recouvrent largement, si bien que rien ne distingue Nantes. La seule estimation qui exclut zéro, la cellule nantaise au-delà de 20 km, irait même à l'inverse de H2, mais c'est un résultat isolé. Quand le lien est faible partout, il reste peu de place pour des écarts entre aires, et H2 reste descriptive.

### 4.4 Qualité ou présence ?

H3 porte sur l'évolution de la part de cadres de 2011 à 2022 dans les 280 communes qui ont un lycée retenu à moins de 5 km. L'estimation utilise les 268 d'entre elles dont au moins un de ces lycées a une valeur ajoutée publiée. Nous régressons cette évolution sur deux mesures, centrées et réduites dans chaque aire, avec les contrôles de la section 2.4, en comparant des communes d'une même aire et d'un même anneau :

- **la qualité** : la valeur ajoutée moyenne des lycées à 5 km, de pente [β~q~]{.grec} ;
- **la présence** : le logarithme du nombre de ces mêmes lycées, de pente [β~n~]{.grec}.

Les intervalles viennent d'un bootstrap par grappes, chaque grappe regroupant les communes qui ont le même lycée le plus proche. H3 est « infirmée » si l'intervalle de la pente de la qualité, ou celui de l'écart entre les deux pentes ([β~q~]{.grec} − [β~n~]{.grec}), est entièrement sous zéro.

In [ ]:
#| label: tbl-fiabilite
#| tbl-cap: "Fiabilité de la valeur ajoutée à 5 km, moitiés 2012-2017 et 2018-2022, Spearman-Brown."
# Fiabilité de l'IVAL : seule la ligne « ensemble » tranche H3 (seuil de 0,5)
fiabilite = h3.fiabilite_ival(base, lycees, collecte.ival_lycees())
tableaux.tableau_fiabilite(fiabilite)

La fiabilité d'ensemble de la valeur ajoutée, 0,608, dépasse le seuil de 0,5, donc H3 est testable. Elle tient surtout à Lyon (0,657) ; à Nantes et à Strasbourg, elle reste sous 0,5.

In [ ]:
#| label: fig-h3
#| fig-cap: >-
#|   Évolution moyenne de la part de cadres de 2011 à 2022 par tiers de valeur ajoutée des
#|   lycées à 5 km, calculés dans chaque aire, communes sans valeur ajoutée à part.
population_h3 = h3.population_h3(base, lycees)
estimation_h3 = h3.estimation_h3(population_h3)
figures.fig_h3(
    h3.terciles_h3(population_h3),
    "La hausse de la part de cadres ne suit pas régulièrement l'IVAL des lycées proches",
)

In [ ]:
#| label: tbl-h3
#| tbl-cap: "Coefficients de H3, en points pour un écart-type, intervalles à 95 %."
# Seule la portée « ensemble » porte le verdict
tableaux.tableau_h3(estimation_h3)

La hausse de la part de cadres n'augmente pas régulièrement avec la valeur ajoutée : elle baisse même légèrement d'un tiers à l'autre à Lyon, monte à Strasbourg avec des intervalles très larges et ne suit aucun ordre à Nantes. Dans la régression (@tbl-h3), c'est le nombre de lycées proches qui accompagne la hausse, bien plus que leur valeur ajoutée : H3 est **« infirmée »**.

::: {.callout-note title="Les chiffres de H3"}
- Hausse de la part de cadres à Lyon, du tiers de valeur ajoutée le plus faible au plus élevé : 5,4, puis 4,6, puis 4,3 points.
- Un écart-type de valeur ajoutée en plus : +0,22 point de hausse [−0,48 ; +0,99].
- Un écart-type de nombre de lycées en plus : +1,73 point [+1,13 ; +2,80].
- Écart entre les deux pentes, [β~q~]{.grec} − [β~n~]{.grec} : −1,51 [−2,85 ; −0,63], entièrement sous zéro.
- Les colonnes par aire sont seulement descriptives : à Nantes et à Strasbourg, la fiabilité de la valeur ajoutée est sous 0,5 (@tbl-fiabilite) et les intervalles sont très larges.
:::

Ce test est défavorable à H3 pour deux raisons. D'abord, l'erreur de mesure de la valeur ajoutée tire sa pente vers zéro, ce qui n'est pas le cas de celle du nombre de lycées. Ensuite, nous comptons les lycées qui existent en 2026 : des lycées ouverts après 2011 dans des communes en croissance gonflent la pente du nombre. Le verdict tient avec d'autres mesures de la qualité (4.5), mais ces variantes ne corrigent aucun de ces deux biais, et l'autocorrélation spatiale (4.5) rend sans doute l'intervalle de l'écart [β~q~ − β~n~]{.grec} trop étroit. Au total, la hausse des cadres va de pair avec le nombre de lycées proches plus qu'avec leur valeur ajoutée mesurée, sans qu'on sache lequel des deux précède l'autre.

### 4.5 Robustesse : figure récapitulative et indice de Moran

Chaque verdict repose sur une seule spécification. Les variantes ci-dessous vérifient qu'il ne dépend pas d'un choix particulier.

#### Pente principale de H1

In [ ]:
#| label: fig-robustesse
#| fig-cap: "Pente principale de H1 : spécification de base et cinq variantes, intervalles à 95 %."
pentes_robustesse = robustesse.robustesse_h1(
    base, paires, lycees, pipeline.lycees_prives_robustesse(), collecte.chefs_lieux()
)
figures.fig_robustesse(
    pentes_robustesse, "Aucune variante ne fait sortir zéro de l'intervalle de la pente principale"
)

Dans toutes les variantes, l'intervalle de la pente principale contient encore zéro, et les verdicts ne changent pas. Deux choix font tout de même bouger les pentes : le contrôle de la densité et celui du prix.

::: {.callout-note title="Ce que changent les variantes" collapse="true"}
- **Sans contrôle de densité**, la pente de Lyon passe de −0,18 à −0,01, et celle de Nantes de −0,15 à −0,29 [−0,67 ; +0,12]. Dans les deux aires, les communes plus denses sont plus proches d'un lycée privé. À distance égale, elles comptent un peu moins de cadres à Lyon et un peu plus à Nantes. Omettre la densité ramène donc la pente de Lyon vers zéro et rend celle de Nantes plus négative, d'où ce contrôle dans la spécification principale (2.4).
- **Avec le prix au m²**, la pente de Nantes passe de −0,15 à −0,05 et celle de Lyon reste presque inchangée (−0,16). À Nantes, les voisines qui diffèrent par l'accès au lycée privé diffèrent donc aussi par le prix. Comme le prix est à la fois cause et conséquence de la part de cadres (2.4), on ne peut pas dire lequel des deux explique l'autre.
- **Avec les lycées professionnels et le privé hors contrat**, les pentes sont les plus négatives (−0,31 à Lyon, −0,24 à Strasbourg), mais leurs intervalles contiennent encore zéro.
- **La pente secondaire de Lyon** change de signe quand le rayon varie entre 3 et 10 km : elle n'apporte rien de stable.
:::

In [ ]:
#| output: asis
# Le détail de toutes les variantes, replié sous la figure
detail = tableaux.tableau_robustesse(pentes_robustesse).to_markdown()
legende = ": Pente de H1 dans chaque variante, en points, intervalle à 95 % {#tbl-robustesse}"
print(tableaux.encadre_replie("Tableau de robustesse détaillé", f"{detail}\n\n{legende}"))

#### Qualité des lycées (H3)

In [ ]:
#| label: tbl-h3-robustesse
#| tbl-cap: "Coefficients de H3 selon la période et l'indicateur de valeur ajoutée, 3 aires."
tableaux.tableau_h3(robustesse.robustesse_h3(base, lycees, collecte.ival_lycees()))

Le verdict tient avec la valeur ajoutée des sessions 2012 à 2025 ([β~q~]{.grec} − [β~n~]{.grec} = −1,39 [−2,63 ; −0,35]) et avec la valeur ajoutée du taux de réussite (−1,91 [−3,45 ; −0,75]).

#### Autocorrélation spatiale

L'indice de Moran mesure si des communes voisines se ressemblent plus que ne le voudrait le hasard. Il est proche de 0 en l'absence de dépendance spatiale et se rapproche de 1 quand les voisines se ressemblent.

In [ ]:
#| label: tbl-moran
#| tbl-cap: "Indice I de Moran de la part de cadres et de l'écart à la moyenne de l'anneau."
tableaux.tableau_moran(robustesse.moran_part_cadres(base))

La part de cadres est très concentrée dans l'espace (indice de 0,33 à 0,61). Une fois retirée la moyenne de chaque anneau, une dépendance nette demeure, surtout à Lyon (0,31). Deux voisines ne sont donc pas deux observations indépendantes, et le bootstrap par commune n'en tient compte qu'en partie : nos intervalles sont probablement trop étroits. Ceux qui contiennent zéro le contiendraient encore s'ils étaient élargis. En revanche, les tailles d'association qu'ils excluent sont moins sûres qu'il n'y paraît, et les intervalles de H3 sont à lire avec prudence.

## 5. Les lycées face aux autres infrastructures

H4 compare le lycée privé à quatre autres infrastructures, avec les mêmes paires, contrôles et tirages que pour H1 : la gare la plus proche, l'APL aux médecins généralistes (accessibilité potentielle localisée, calculée par la DREES en consultations accessibles par habitant), et le nombre d'équipements sportifs et culturels à 5 km. Chaque mesure est centrée et réduite dans l'aire, et orientée de sorte qu'une pente positive signifie « plus d'accès, plus de cadres ».

In [ ]:
#| label: fig-h4
#| fig-cap: >-
#|   Pente des paires contiguës pour un écart-type d'accès en plus, par infrastructure et par
#|   aire, avec intervalles à 95 % par bootstrap par commune.
pentes_h4 = h1.pentes_h4(base, paires)
figures.fig_h4(
    pentes_h4, "Lycées, gares, médecins, sport, culture : aucune infrastructure ne se détache"
)

Les quinze pentes sont petites, et leurs intervalles, larges, se recouvrent tous. Celles du lycée privé vont de +0,43 à +0,90 point par écart-type selon l'aire, sans qu'aucun intervalle n'exclue zéro. Certaines pentes d'autres infrastructures sont plus fortes (installations sportives à Lyon, gare à Nantes), mais seule celle de la gare à Nantes exclut zéro (+1,64 [+0,12 ; +3,25]). Une estimation sur quinze, c'est à peu près ce que produirait le hasard. Nous ne concluons donc pas que le lien est plus fort avec les lycées : aucune infrastructure ne se détache.

## 6. Conclusion

In [ ]:
#| label: tbl-hypotheses
#| tbl-cap: "Verdict de chaque hypothèse."
verdicts_h1 = h1.verdict_h1(pentes_principales, pentes_secondaires, anneaux)
verdict_h3 = h3.verdict_h3(estimation_h3, fiabilite.loc["ensemble", "r_spearman_brown"])
tableaux.tableau_hypotheses(verdicts_h1, pentes_principales, verdict_h3)

Entre communes voisines, être plus près d'un lycée privé ne s'accompagne pas d'un écart net de part de cadres. Le résultat nul est établi à Lyon, et les données ne permettent pas de conclure à Nantes et à Strasbourg (H1). L'aire de l'Ouest ne se distingue pas des deux autres (H2). À Lyon, ce résultat nul n'exclut cependant pas un écart de quelques points, et son intervalle est sans doute trop étroit. La hausse des cadres entre 2011 et 2022 va davantage de pair avec le nombre de lycées proches qu'avec leur valeur ajoutée (H3 infirmée), et aucune infrastructure ne se détache des autres (H4).

À l'échelle de la commune, une fois la position dans l'aire prise en compte, l'accès au lycée privé ne distingue donc pas nettement les communes riches en cadres de leurs voisines. Le contraste le plus net reste celui entre le centre et la périphérie, présent à chaque recensement (@tbl-gradient). Cela n'empêche pas que des familles choisissent leur logement en fonction du lycée. Mais un lycée privé dessert plusieurs communes voisines à la fois, et la commune est sans doute une maille trop large pour observer ce choix.

Les limites présentées en introduction s'appliquent ici : causalité inverse, sélection résidentielle, croissance démographique et distances à vol d'oiseau. S'y ajoutent le faible nombre de communes exposées à Strasbourg et des intervalles probablement trop étroits. La carte exploratoire ci-dessous illustre le problème de la maille.

In [ ]:
#| output: asis
parts_iris = nettoyage.part_cadres_iris(
    collecte.activite_iris_2011(), collecte.activite_iris_2022(), collecte.passage_communes()
)
# Le zoom porte sur la commune-centre ; à Lyon, 69123, le code des contours IRIS
centres = geo.communes_centres(base)
cartes_iris = {
    aire: figures.carte_iris(
        collecte.contours_iris(code_aav), parts_iris, centres[aire], f"Aire de {aire}"
    )
    for aire, code_aav in config.AIRES_RETENUES.items()
}
legende = (
    "**Dans les villes-centres, la part de cadres va de moins de 10 % à plus de 50 "
    "% selon l'IRIS.** Part des cadres parmi les actifs de 15 à 64 ans par IRIS en "
    "2011 et 2022, sur l'aire entière et autour de la commune-centre, une aire par "
    "onglet. Carte exploratoire, sans test ni lycées."
)
afficher_onglets(cartes.onglets_par_aire("fig-iris", cartes_iris, legende))

À l'intérieur des villes-centres, des quartiers (les IRIS de l'Insee, d'environ 2 000 habitants) de moins de 10 % de cadres côtoient des quartiers qui en comptent 50 % ou plus, et les quartiers les plus foncés sont plus nombreux en 2022 qu'en 2011. Une moyenne communale mélange donc des quartiers très différents. La carte porte sur les 15-64 ans, seule tranche d'âge disponible à cette échelle.

Quatre pistes permettraient de prolonger ce travail :

- **l'échelle du quartier**, avec la même méthode de paires ;
- **les migrations résidentielles** du recensement, pour voir où s'installent les cadres qui arrivent et répondre en partie à la question de la causalité inverse ;
- **les temps de trajet**, à la place des distances à vol d'oiseau ;
- **les collèges**, au cœur du mécanisme décrit par Fack et Grenet. À 5 km, ils donneraient environ deux fois plus de paires de voisines de statut différent (279 à Lyon, 113 à Nantes et 53 à Strasbourg, contre 169, 62 et 26 pour les lycées), et l'indice de position sociale des collèges comme la carte scolaire des collèges publics sont en open data.

## 7. Annexes

### Références

- Black, S. E. (1999). « Do Better Schools Matter? Parental Valuation of Elementary Education ». *Quarterly Journal of Economics*, 114(2), p. 577-599.
- Fack, G. et Grenet, J. (2010). « When do better schools raise housing prices? Evidence from Paris public and private schools ». *Journal of Public Economics*, 94(1-2), p. 59-77.
- DEPP, *Repères et références statistiques sur les enseignements, la formation et la recherche*, ministère de l'Éducation nationale, édition 2025.
- Mouvement lycéen de l'automne 2026 : bilan du ministère de l'Intérieur du 2 octobre 2026 (1 027 lycées concernés sur environ 3 700), repris dans l'article « Mouvement lycéen et étudiant de 2026 en France », *Wikipédia*, consulté le 15 novembre 2026.
- Insee, recensements de la population 1999 à 2022 ; DEPP, IPS et IVAL des lycées ; sources détaillées en 2.1 et dans l'annexe ci-dessous.

### Reproduire le projet

Le projet demande Python 3.13, [uv](https://docs.astral.sh/uv/) et [Quarto](https://quarto.org/) (déjà installé sur le SSPCloud). Aucune variable d'environnement n'est nécessaire.

```bash
uv sync --locked      # exactement les dépendances figées dans uv.lock
uv run quarto render  # exécute rapport.ipynb et écrit le site dans _site/
```

Les sources qui peuvent changer sans prévenir sont lues dans des copies gelées publiques, publiées dans une release GitHub du dépôt. Les fichiers de l'Insee sont lus à leur URL d'origine. Le `sha256` de chaque fichier, listé dans le @tbl-fichiers, est vérifié à chaque lecture. Sur un clone neuf, sans identifiants S3, l'exécution complète prend environ 16 minutes, téléchargements compris (mesure du 04/11/2026).

In [ ]:
#| label: tbl-fichiers
#| tbl-cap: "Fichiers bruts lus par le rapport et leur sha256, contrôlé à chaque lecture."
pipeline.fichiers_bruts()

### Dictionnaire des variables

In [ ]:
#| output: asis
# Replié : le dictionnaire sert de référence, pas de lecture suivie
print(
    tableaux.encadre_replie(
        "Dictionnaire des variables", tableaux.document_markdown(config.CHEMIN_DICTIONNAIRE)
    )
)

### Décisions prises après les premiers résultats

Quelques choix ont été faits après avoir vu les premières estimations de H1 et de H4. Nous les listons pour que le lecteur puisse juger les résultats en sachant quand ils ont été faits :

- **H4 reste descriptive**, comme H2, car aucun critère de verdict n'avait été écrit avant son estimation. Nous avons aussi abandonné sa comparaison par anneaux, qui suppose de classer les communes en exposées et non exposées et ne s'applique donc pas à des mesures continues.
- **Sport et culture dans H4** : nous utilisons log(1 + nombre d'équipements à 5 km), car la distance à l'équipement le plus proche ne distingue pas les communes.
- **Prix au m²** : nous utilisons la période de juillet 2021 à juin 2026, la seule publiée, et la robustesse ne porte que sur Lyon et Nantes, car l'Alsace-Moselle est absente de DVF.
- **Lycées agricoles** : ils sont exclus de toutes les variantes, car leur contrat relève du ministère de l'Agriculture.
- **Détails d'exécution** : code des anneaux et de H3, tirages écartés du bootstrap des paires, écarts à la moyenne d'anneau pour l'indice de Moran, définition de la commune-centre exclue, intervalles des tiers de la @fig-h3.
- **Robustesses prévues dès le départ mais codées après ces premiers résultats** : distances depuis la mairie, valeur ajoutée 2012-2025 et valeur ajoutée du taux de réussite.